In [12]:
%%writefile lab8_work.py

import json
import re
import statistics
import lab8_llm as llm
from lab8_llm import ask

llm.set_seed(0)

print(ask("Say hello in one short sentence."))

call = llm.LOG[-1]

print(
    "calls so far:", len(llm.LOG),
    "| seconds:", call["seconds"],
    "| reply length:", len(call["reply"])
)

Overwriting lab8_work.py


In [13]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "lab8_llm.py":
            print("FOUND:", os.path.join(root, file))

FOUND: /kaggle/input/datasets/abdulsamad010/labno08/lab8_llm.py


In [14]:
import shutil

shutil.copy(
    "/kaggle/input/datasets/abdulsamad010/labno08/lab8_llm.py",
    "/kaggle/working/lab8_llm.py"
)

print("File copied successfully!")

File copied successfully!


In [15]:
%run lab8_work.py

Hello from the Lab 8 stand-in. I am a Python function, not a language model.
calls so far: 1 | seconds: 0.0 | reply length: 76


In [25]:
import lab8_llm as llm
from lab8_llm import ask

QUESTION = "Why does the game crash when the player opens the inventory?"

CONTEXT = {
    "log": "Error log: " + llm.LOG_TEXT.replace("\n", " "),
    "script": "Script line: 42: label.text = slot.item.name;",
    "version": "Engine version: Unity 2022.3 LTS"
}

def build_prompt(question, items):
    parts = [CONTEXT[name] for name in items]
    return "\n".join(parts + ["Question: " + question])


for items in [[], ["log"]]:
    prompt = build_prompt(QUESTION, items)
    reply = ask(prompt)

    print(f"context {items or 'none'}: {len(prompt)} characters sent")
    print("reply:", reply)
    print("----------------------")
    
def clean_json(text):
    """Remove a JSON code fence if the model added one."""
    text = text.strip()

    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]

    return text.strip()


def check_plan(text):
    """Check whether the model returned a usable plan."""
    try:
        plan = json.loads(clean_json(text))
    except json.JSONDecodeError:
        return None, "not valid JSON"

    if not isinstance(plan, list) or not 3 <= len(plan) <= 6:
        return None, "needs 3 to 6 steps"

    if not all(
        isinstance(s, dict) and {"step", "check"} <= s.keys()
        for s in plan
    ):
        return None, "a step is missing 'step' or 'check'"

    return plan, ""


def plan_prompt(goal):
    return (
        f"Goal: {goal}\n"
        "Split the goal into 3 to 6 steps. Reply with a JSON array only. "
        'Each item must look like {"step": "...", '
        '"check": "how to know it is done"}.'
    )


for seed in (0, 1):
    llm.set_seed(seed)

    plan, problem = check_plan(
        ask(plan_prompt("Create a simple enemy AI prototype"))
    )

    print(f"seed {seed}: problem = {problem or 'none'}")

    for i, item in enumerate(plan or [], 1):
        print(
            f" {i}. {item['step']} -> check: {item['check']}"
        )

    

context none: 70 characters sent
reply: I cannot be sure without more information. Common causes of a crash when a menu opens are null references, missing assets or endless loops.
----------------------
context ['log']: 223 characters sent
reply: Likely cause: a NullReferenceException at InventoryUI.cs:42, so something on that line is empty.
----------------------
seed 0: problem = none
 1. Define behaviour -> check: Patrol, chase and retreat are written down
 2. Choose states -> check: Each behaviour is a named state
 3. Implement logic -> check: Events switch the enemy between states
 4. Test scenarios -> check: Player distance and obstacles are tested
seed 1: problem = not valid JSON


In [26]:
import json
import re
import statistics
import lab8_llm as llm
from lab8_llm import ask

In [45]:


# LAB TASK 1 - ZERO-SHOT VS FEW-SHOT
# With Accuracy, Precision, Recall and F1-score

import re
import lab8_llm as llm

from lab8_llm import ask

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# 1. Bug reports and their correct categories

REPORTS = [
    ("Player falls through the floor after a jump", "Physics / Collision"),
    ("Enemy walks through a wall", "Physics / Collision"),
    ("Game freezes for five seconds when loading level 3", "Performance"),
    ("Frame rate drops to 12 fps in the forest scene", "Performance"),
    ("Textures take a long time to appear", "Performance"),
    ("Boss health bar is not shown on the screen", "UI"),
    ("Inventory button does not respond when clicked", "UI"),
    ("Score resets to zero after pausing", "Game Logic"),
    ("Enemy keeps patrolling after the player is dead", "Game Logic"),
    ("Door stays closed even after the key is collected", "Game Logic"),
    ("Footstep sound plays twice for each step", "Audio"),
    ("Music stops when the menu is opened", "Audio"),
]


# 2. Available categories

CATEGORIES = "Physics / Collision, Performance, UI, Game Logic, Audio"


# 3. Zero-shot prompt (without examples)

ZERO = (
    "Classify this bug report as one of: " + CATEGORIES + ". "
    "Also give a severity (Low, Medium or High) and one action.\n"
    "Report: {report}"
)


# 4. Few-shot prompt (with examples)

FEW = (
    "Classify bug reports. Use exactly the format: Category | Severity | Action\n"
    "Input: Player falls through the floor\n"
    "Output: Physics / Collision | High | Check collider and collision layer\n"
    "Input: Loading screen freezes the game\n"
    "Output: Performance | Medium | Profile the scene and check asset sizes\n"
    "Input: {report}\nOutput:"
)


# 5. Check the output format

PATTERN = re.compile(
    r"^(Physics / Collision|Performance|UI|Game Logic|Audio) "
    r"\| (Low|Medium|High) \|(.+)$"
)


def parse(reply):

    m = PATTERN.match(reply.strip())

    if m:
        return (m.group(1), m.group(2), m.group(3))

    return None


# 6. Run one complete test

def run_method(template, seed):

    llm.set_seed(seed)

    valid = 0
    correct = 0

    actual = []
    predicted = []

    for report, truth in REPORTS:

        reply = ask(template.format(report=report))

        parsed = parse(reply)

        if parsed:

            valid += 1

            actual.append(truth)
            predicted.append(parsed[0])

            if parsed[0] == truth:
                correct += 1

    return valid, correct, actual, predicted


# 7. Compare zero-shot and few-shot over 10 seeds

for name, template in [
    ("zero-shot", ZERO),
    ("few-shot", FEW)
]:

    results = [
        run_method(template, seed)
        for seed in range(10)
    ]

    valid = sum(v for v, c, a, p in results)

    correct = sum(c for v, c, a, p in results)

    total = 10 * len(REPORTS)

    # Combine actual and predicted labels

    actual = []
    predicted = []

    for v, c, a, p in results:

        actual.extend(a)
        predicted.extend(p)

    print("\n==============================")
    print(name.upper())
    print("==============================")

    # Original lab measurements

    print(
        f"Format Valid: {valid}/{total} "
        f"({valid / total:.2%})"
    )

    print(
        f"Correct Among Valid: {correct}/{valid} "
        f"({correct / valid:.2%})"
        if valid > 0 else "Correct Among Valid: 0%"
    )

    print(
        f"Correct Overall: {correct}/{total} "
        f"({correct / total:.2%})"
    )

    # New evaluation metrics

    if valid > 0:

        print("\n--- Evaluation Metrics ---")

        print(
            "Accuracy:",
            round(accuracy_score(actual, predicted), 4)
        )

        print(
            "Precision:",
            round(precision_score(
                actual,
                predicted,
                average="macro",
                zero_division=0
            ), 4)
        )

        print(
            "Recall:",
            round(recall_score(
                actual,
                predicted,
                average="macro",
                zero_division=0
            ), 4)
        )

        print(
            "F1 Score:",
            round(f1_score(
                actual,
                predicted,
                average="macro",
                zero_division=0
            ), 4)
        )

    else:

        print("No valid predictions. Metrics cannot be calculated.")




ZERO-SHOT
Format Valid: 49/120 (40.83%)
Correct Among Valid: 34/49 (69.39%)
Correct Overall: 34/120 (28.33%)

--- Evaluation Metrics ---
Accuracy: 0.6939
Precision: 0.6912
Recall: 0.6949
F1 Score: 0.6836

FEW-SHOT
Format Valid: 115/120 (95.83%)
Correct Among Valid: 107/115 (93.04%)
Correct Overall: 107/120 (89.17%)

--- Evaluation Metrics ---
Accuracy: 0.9304
Precision: 0.928
Recall: 0.9371
F1 Score: 0.9293


In [28]:

# LAB TASK 2 - HOW MUCH CONTEXT IS ENOUGH?

# Score the model's answer
def score_answer(reply):
    reply = reply.lower()

    if "nullreferenceexception" in reply and "slot.item" in reply:
        return 2

    elif "nullreferenceexception" in reply:
        return 1

    else:
        return 0


# Test different context combinations
for items in [
    [],
    ["log"],
    ["log", "script"],
    ["log", "script", "version"]
]:

    prompt = build_prompt(QUESTION, items)

    llm.set_seed(0)

    reply = ask(prompt)

    score = score_answer(reply)

    print(
        f"context {items or 'none'} "
        f"characters {len(prompt)} "
        f"score {score}"
    )


context none characters 70 score 0
context ['log'] characters 223 score 1
context ['log', 'script'] characters 269 score 2
context ['log', 'script', 'version'] characters 302 score 2


In [29]:

# LAB TASK 3 - PLAN, VALIDATE AND RETRY

def get_plan(goal, max_retries=2):

    prompt = plan_prompt(goal)

    for attempt in range(max_retries + 1):

        # Ask the model
        reply = ask(prompt)

        # Validate the response
        plan, problem = check_plan(reply)

        # If plan is valid, return it
        if plan is not None:
            return plan, attempt + 1

        # If invalid, tell the model what went wrong
        prompt += (
            f"\nYour previous answer was rejected because: {problem}. "
            "Please correct the problem and return a valid JSON array."
        )

    return None, max_retries + 1


# Three goals
GOALS = [
    "Create a simple enemy AI prototype",
    "Prepare a playable demo for Friday",
    "Add a save system to the game"
]


# Counters
first_try_ok = 0
retries_used = 0
failures = 0

# Clear previous model call records
llm.LOG.clear()

# Run 20 seeds with 3 goals each
for seed in range(20):

    llm.set_seed(seed)

    for goal in GOALS:

        plan, attempts = get_plan(goal)

        if attempts == 1 and plan is not None:
            first_try_ok += 1

        retries_used += attempts - 1

        if plan is None:
            failures += 1


# Calculate total requests
total = 20 * len(GOALS)

# Print results
print("plans asked for:", total)

print(
    f"valid on the first try: {first_try_ok} "
    f"({first_try_ok / total:.0%})"
)

print("retries used:", retries_used)

print("gave up after retries:", failures)

print("model calls in total:", len(llm.LOG))


plans asked for: 60
valid on the first try: 40 (67%)
retries used: 22
gave up after retries: 0
model calls in total: 82


In [43]:


# LAB TASK 4 - A REACT DEBUGGING AGENT
# TAKE-HOME EXTENSION 3 - TOOL DESIGN

import re
import lab8_llm as llm
from lab8_llm import ask


# 1. Files available to the agent

FILES = {
    "InventoryUI.cs": """40: foreach (var slot in slots)
41: {
42: label.text = slot.item.name;
43: }""",

    "PlayerController.cs": "Player movement script",
    "GameManager.cs": "Game management script",
    "EnemyAI.cs": "Enemy AI script"
}


# 2. Safe tools

TOOLS = {
    "read_log": lambda arg: llm.LOG_TEXT,

    "search_file": lambda arg: FILES.get(
        arg, "file not found: " + arg
    ),

    "check_version": lambda arg: "Unity 2022.3 LTS",

    # NEW FOURTH TOOL
    "list_files": lambda arg: ", ".join(FILES.keys())
}


# 3. Instructions for the model

HEADER = (
    "You are a debugging agent. Available tools: "
    "read_log[], search_file[name], check_version[], list_files[].\n"
    "Reply with 'Thought: ...' and then either "
    "'Action: tool[argument]' or 'Final: answer'.\n"
)


# 4. Parse model response

def parse_step(reply):

    final = re.search(r"Final:\s*(.+)", reply, re.S)

    if final:
        return ("final", final.group(1).strip())

    act = re.search(r"Action:\s*(\w+)\[(.*?)\]", reply)

    if act:
        return ("action", act.group(1), act.group(2))

    return ("bad", reply)


# 5. ReAct agent loop

def react(question, max_steps=5):

    transcript = HEADER + "Question: " + question + "\n"

    trace = []

    for step in range(1, max_steps + 1):

        # Ask the model
        reply = ask(transcript)

        # Understand the response
        kind = parse_step(reply)

        # CASE 1: Model gives final answer
        if kind[0] == "final":

            answer = kind[1]

            trace.append(
                f"step {step}: Final -> {answer}"
            )

            return answer, trace

        # CASE 2: Model gives incorrect format
        elif kind[0] == "bad":

            observation = (
                "Invalid format. Use Thought followed by "
                "Action or Final."
            )

            trace.append(
                f"step {step}: bad reply -> {observation}"
            )

        # CASE 3 and 4: Model requests a tool
        else:

            tool = kind[1]
            arg = kind[2]

            # Check whether tool exists
            if tool not in TOOLS:

                observation = (
                    f"unknown tool {tool}. Allowed: "
                    + ", ".join(TOOLS.keys())
                )

            # Run valid tool
            else:

                observation = TOOLS[tool](arg)

            trace.append(
                f"step {step}: {tool}[{arg}] -> {observation}"
            )

        # Add model response and observation
        transcript += (
            reply + "\nObservation: " + observation + "\n"
        )

    # Stop after maximum steps
    return None, trace + ["stopped: step limit reached"]


# 6. Test the debugging agent

QUESTION4 = "Why does the game crash when the inventory opens?"

llm.set_seed(2)

answer, trace = react(QUESTION4)

print("\n--- REACT DEBUGGING AGENT ---")
print("\n".join(trace))
print("answer:", answer)


# 7. Test 30 normal runs

outcomes = {
    "solved": 0,
    "step limit": 0
}

unknown_tool_seen = 0

for seed in range(30):

    llm.set_seed(seed)
    llm.LOG.clear()

    answer, trace = react(QUESTION4)

    if answer:
        outcomes["solved"] += 1
    else:
        outcomes["step limit"] += 1

    if any("unknown tool" in t for t in trace):
        unknown_tool_seen += 1


print("\n--- 30 NORMAL RUNS ---")

print(
    "30 normal runs:",
    outcomes,
    "| runs that tried an unknown tool:",
    unknown_tool_seen
)


# 8. Test stuck model

llm.set_seed(0, mode="stuck")

answer, trace = react(QUESTION4)

print("\n--- STUCK MODEL TEST ---")

print(
    "stuck model:",
    answer,
    "|",
    trace[-1],
    "| steps:",
    len(trace) - 1
)


# 9. TAKE-HOME EXTENSION 3
# Test the new list_files tool

print("\n--- TAKE-HOME: LIST FILES TOOL ---")

print("Available project files:")
print(TOOLS["list_files"](""))


# 10. Three test questions

TEST_QUESTIONS = [
    "What files are available in the project?",
    "Show me the names of all project scripts.",
    "Which files can I inspect to debug the player movement?"
]

print("\n--- THREE TEST QUESTIONS ---")

for question in TEST_QUESTIONS:

    print("\nQuestion:", question)

    # Test the new tool directly
    result = TOOLS["list_files"]("")

    print("Available files:", result)



--- REACT DEBUGGING AGENT ---
step 1: read_log[] -> ERROR NullReferenceException: Object reference not set to an instance of an object
  at InventoryUI.OpenInventory() in InventoryUI.cs:line 42
step 2: search_file[InventoryUI.cs] -> 40: foreach (var slot in slots)
41: {
42: label.text = slot.item.name;
43: }
step 3: run_game[] -> unknown tool run_game. Allowed: read_log, search_file, check_version, list_files
step 4: run_game[] -> unknown tool run_game. Allowed: read_log, search_file, check_version, list_files
step 5: Final -> In InventoryUI.cs line 42, check that slot.item is not null before reading name.
answer: In InventoryUI.cs line 42, check that slot.item is not null before reading name.

--- 30 NORMAL RUNS ---
30 normal runs: {'solved': 30, 'step limit': 0} | runs that tried an unknown tool: 3

--- STUCK MODEL TEST ---
stuck model: None | stopped: step limit reached | steps: 5

--- TAKE-HOME: LIST FILES TOOL ---
Available project files:
InventoryUI.cs, PlayerController.cs, Game

In [44]:
!python lab8_work.py

Hello from the Lab 8 stand-in. I am a Python function, not a language model.
calls so far: 1 | seconds: 0.0 | reply length: 76


In [31]:
# LAB TASK 5 - CHECK GENERATED CODE

TRUSTED = {
    "Physics.Raycast",
    "transform.Translate",
    "Destroy",
    "audioSource.Play",
    "transform.Find"
}

TASKS = [
    "cast a ray forward from the player",
    "move the character with its speed",
    "destroy the object after two seconds",
    "play the sound attached to this object",
    "find the child object called Weapon"
]

In [33]:
def called_names(line):
    return re.findall(
        r"([A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*)\s*\(",
        line
    )


def unsupported(line):
    return [
        name for name in called_names(line)
        if name not in TRUSTED
    ]

In [37]:
llm.set_seed(3)

for task in TASKS:
    line = ask(f"Write one line of C# to {task}.")
    bad = unsupported(line)

    if bad:
        print(f"UNSUPPORTED {bad} | {line}")
    else:
        print(f"supported | {line}")

UNSUPPORTED ['Physics.RaycastSmart'] | Physics.RaycastSmart(origin, direction);
supported | transform.Translate(move * speed * Time.deltaTime);
supported | Destroy(gameObject, 2f);
supported | audioSource.Play();
supported | transform.Find("Weapon");


In [36]:
lines_checked = 0
flagged = 0
wrong_destroy_not_flagged = 0

for seed in range(40):
    llm.set_seed(seed)

    for task in TASKS:
        line = ask(f"Write one line of C# to {task}.").strip()
        lines_checked += 1

        if unsupported(line):
            flagged += 1

        if line == "Destroy(2f, gameObject);" and not unsupported(line):
            wrong_destroy_not_flagged += 1

print(
    f"lines checked: {lines_checked}, "
    f"flagged as unsupported: {flagged} "
    f"({flagged / lines_checked:.1%})"
)

print(
    "real function used wrongly and NOT flagged:",
    wrong_destroy_not_flagged
)

lines checked: 200, flagged as unsupported: 54 (27.0%)
real function used wrongly and NOT flagged: 2
